In [9]:

import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(['i', 'on', 'the', 'like', 'to', 'and', 'in', 'over'])
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    tokens = text.split()
    cleaned_tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(cleaned_tokens)
    
dataset = ["I love playing football on the weekends",
 "I enjoy hiking and camping in the mountains",
 "I like to read books and watch movies",
 "I prefer playing video games over sports",
 "I love listening to music and going to concerts"]
cleaned_dataset = [preprocess_text(doc) for doc in dataset]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_dataset)

k = 2 # Define the number of clusters
km = KMeans(n_clusters=k)
km.fit(X)
# Predict the clusters for each document
y_pred = km.predict(X)
# Display the document and its predicted cluster in a table
table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))

# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
 print("Cluster %d:" % i)
 for ind in order_centroids[i, :10]:
  print(' %s' % terms[ind])
 print()

# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)


"""
The purity after text preprocessing:0.8 whereas before it is: 0.6.
There is an increase of 0.2 which means its more accurate where overlapping 
words that does not hold meaning are removed for a more accurate clustering
"""

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        1
I like to read books and watch movies                              0
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    0

Top terms per cluster:
Cluster 0:
 love
 playing
 weekend
 football
 book
 read
 watch
 movie
 sport
 video

Cluster 1:
 hiking
 enjoy
 camping
 mountain
 sport
 video
 watch
 weekend
 music
 read

Purity: 0.8


'\nThe purity after text preprocessing:0.8 whereas before it is: 0.6.\nThere is an increase of 0.2 which means its more accurate where overlapping \nwords that does not hold meaning are removed for a more accurate clustering\n'